In [10]:
import pandas as pd
import numpy as np

df = pd.read_excel('Metro_Interstate_Traffic_Volume.xlsx')

df['date_time'] = pd.to_datetime(df['date_time'])
df = df.sort_values('date_time')

df.head()

,holiday,temp,rain_1h,snow_1h,clouds_all,weather_main,weather_description,date_time,traffic_volume
0,NaN,288.28,0.0,0.0,40,Clouds,scattered clouds,2012-10-02 09:00:00,5545
1,NaN,289.36,0.0,0.0,75,Clouds,broken clouds,2012-10-02 10:00:00,4516
2,NaN,289.58,0.0,0.0,90,Clouds,overcast clouds,2012-10-02 11:00:00,4767
3,NaN,290.13,0.0,0.0,90,Clouds,overcast clouds,2012-10-02 12:00:00,5026
4,NaN,291.14,0.0,0.0,75,Clouds,broken clouds,2012-10-02 13:00:00,4918


In [11]:
# Aggregate numeric columns
df_num = df.groupby('date_time').mean(numeric_only=True)

# Keep categorical separately
df_cat = df.drop_duplicates('date_time')[['date_time','weather_main','holiday']]
df_cat = df_cat.set_index('date_time')

# Combine
df = df_num.join(df_cat)

df = df.sort_index()
print("Rows after dedup:", len(df))

Rows after dedup: 40575


In [12]:
# Holiday encoding
df['holiday'] = df['holiday'].fillna('None')

# One-hot encoding
df = pd.get_dummies(df, columns=['weather_main','holiday'], drop_first=True)

df.head()

,temp,rain_1h,snow_1h,clouds_all,traffic_volume,weather_main_Clouds,weather_main_Drizzle,weather_main_Fog,weather_main_Haze,weather_main_Mist,...,holiday_Independence Day,holiday_Labor Day,holiday_Martin Luther King Jr Day,holiday_Memorial Day,holiday_New Years Day,holiday_None,holiday_State Fair,holiday_Thanksgiving Day,holiday_Veterans Day,holiday_Washingtons Birthday
date_time,,,,,,,,,,,,,,,,,,,,,
2012-10-02 09:00:00,288.28,0.0,0.0,40.0,5545.0,True,False,False,False,False,...,False,False,False,False,False,True,False,False,False,False
2012-10-02 10:00:00,289.36,0.0,0.0,75.0,4516.0,True,False,False,False,False,...,False,False,False,False,False,True,False,False,False,False
2012-10-02 11:00:00,289.58,0.0,0.0,90.0,4767.0,True,False,False,False,False,...,False,False,False,False,False,True,False,False,False,False
2012-10-02 12:00:00,290.13,0.0,0.0,90.0,5026.0,True,False,False,False,False,...,False,False,False,False,False,True,False,False,False,False
2012-10-02 13:00:00,291.14,0.0,0.0,75.0,4918.0,True,False,False,False,False,...,False,False,False,False,False,True,False,False,False,False


In [13]:
df['hour_sin'] = np.sin(2 * np.pi * df.index.hour / 24)
df['hour_cos'] = np.cos(2 * np.pi * df.index.hour / 24)

df['dow_sin']  = np.sin(2 * np.pi * df.index.dayofweek / 7)
df['dow_cos']  = np.cos(2 * np.pi * df.index.dayofweek / 7)

df['month_sin']= np.sin(2 * np.pi * df.index.month / 12)
df['month_cos']= np.cos(2 * np.pi * df.index.month / 12)

In [14]:
df['lag_1'] = df['traffic_volume'].shift(1)
df['lag_2'] = df['traffic_volume'].shift(2)
df['lag_24'] = df['traffic_volume'].shift(24)

df = df.dropna(subset=['lag_1', 'lag_2', 'lag_24'])

In [15]:
TARGET = 'traffic_volume'
FEATURES = [c for c in df.columns if c != TARGET]

train = df[df.index.year <= 2017]
test  = df[df.index.year >= 2018]

print(f"Train: {len(train)} ({train.index.min()} -> {train.index.max()})")
print(f"Test:  {len(test)} ({test.index.min()} -> {test.index.max()})")

Train: 34018 (2012-10-03 12:00:00 -> 2017-12-31 23:00:00)
Test:  6533 (2018-01-01 00:00:00 -> 2018-09-30 23:00:00)


In [16]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train = scaler.fit_transform(train[FEATURES])
X_test  = scaler.transform(test[FEATURES])

y_train = train[TARGET].values
y_test  = test[TARGET].values

In [17]:
with pd.ExcelWriter("Metro_Traffic_Splits.xlsx") as writer:
    train.to_excel(writer, sheet_name="Train")
    test.to_excel(writer, sheet_name="Test")

print("Train/Test splits saved!")
output_file = "Metro_Traffic_Processed.xlsx"

df.to_excel(output_file)

print(f"File saved as: {output_file}")

Train/Test splits saved!
File saved as: Metro_Traffic_Processed.xlsx


In [41]:
# Train Random Forest Regressor

from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import OneHotEncoder

def load_data():
    df = pd.read_excel('Metro_Traffic_Processed.xlsx', index_col=0, parse_dates=True)
    X_features = df.drop(columns=['traffic_volume'])
    y_target = df['traffic_volume']
    return X_features, y_target

def build_features(X, y, lags=[1, 3, 24]):
    for lag in lags:
        X[f'lag_{lag}'] = y.shift(lag)
    X = X.dropna()
    y = y.loc[X.index]
    return X, y

def train_random_forest(X_features, y_target, X_train, y_train):
    cat_cols = X_features.select_dtypes(include=['object', 'category']).columns.tolist()

    preprocessor = ColumnTransformer(
        transformers=[
            ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols),
        ],
        remainder='passthrough'
    )
 
    pipeline = Pipeline(steps=[
        ('preproc', preprocessor),
            ('rf', RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1))
    ])
    pipeline.fit(X_train, y_train)
    return pipeline

In [42]:
# Test random forest training

from sklearn.metrics import root_mean_squared_error


X_features, y_target = load_data()
X_features, y_target = build_features(X_features, y_target)

# Filter train/test to match the indices after build_features removed NaNs
X_train_df = X_features[X_features.index.year <= 2017]
y_train_df = y_target[y_target.index.year <= 2017]
X_test_df = X_features[X_features.index.year >= 2018]
y_test_df = y_target[y_target.index.year >= 2018]

pipeline = train_random_forest(X_features, y_target, X_train_df, y_train_df)
y_pred = pipeline.predict(X_test_df)
mae = mean_absolute_error(y_test_df, y_pred)
rmse = root_mean_squared_error(y_test_df, y_pred)
r2 = r2_score(y_test_df, y_pred)

print(f"MAE: {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"Normalized RMSE: {(rmse)/y_test_df.mean():.4f}")
print(f"R2 Score: {r2:.4f}")



c:\CISC484\FinalProject\.venv\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
c:\CISC484\FinalProject\.venv\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
c:\CISC484\FinalProject\.venv\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
c:\CISC484\FinalProject\.venv\Lib\site-packages\sklearn\utils\parallel.py:144: UserWa

MAE: 145.93
RMSE: 234.36
Normalized RMSE: 0.0705
R2 Score: 0.9859


c:\CISC484\FinalProject\.venv\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
c:\CISC484\FinalProject\.venv\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
c:\CISC484\FinalProject\.venv\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
c:\CISC484\FinalProject\.venv\Lib\site-packages\sklearn\utils\parallel.py:144: UserWa